# Regression with random data

Replicating results from:
>Alpert, William T., Kenneth A. Couch, and Oskar R. Harmon. 2016. [“A Randomized Assessment of Online Learning”](https://www.aeaweb.org/articles?id=10.1257/aer.p20161057). American Economic Review, 106 (5): 378-82. 

Replication file available [here](https://www.openicpsr.org/openicpsr/project/113462/version/V1/view).
* This replication file includes .do file and replication data. Copies are available in this repository:
  * Do file: `misc/4_Replication-Program3.do`
  * Data file: `ReplicationData2.dta`
* CSV of relevant data also available [here](https://github.com/matheusfacure/python-causality-handbook/blob/master/causal-inference-for-the-brave-and-true/data/online_classroom.csv)

This example was motivated by [this](https://matheusfacure.github.io/python-causality-handbook/05-The-Unreasonable-Effectiveness-of-Linear-Regression.html) website.

In [1]:
import pandas as pd
import numpy as np

import os

# One way of running regressions
import statsmodels.formula.api as smf

# helpful for formatting
from tabulate import tabulate  # makes nice plain text tables from dataframes
import textwrap                # allows you to wrap text in a string (so it's not too long)

In [2]:
ROOTDIR = os.path.dirname(os.getcwd())
DATADIR = os.path.join(ROOTDIR, 'data')

Note that the replication is completed in Stata. I'm going to run several regressions in python that match the output.
In particular, I'm interested in replicating Table 1. I know the Stat code that does this is:


<img src="img/ACH16-Alpert_Couch_Harmon[2016]-AER-PnP-Table1.png" width=300  />

Looking at the paper and replication file, it seems like the first few regressions in the replication file do this:

```Stata
eststo out1: reg falsexam format_ol format_blended if enroll_count==3
eststo out2: reg falsexam format_ol format_blended gpa cum_total_crds sat_math_NEW sat_verbal_NEW if enroll_count==3
eststo out3: reg falsexam format_ol format_blended gpa cum_total_crds sat_math_NEW sat_verbal_NEW gender asian black hispanic hawaiian unknown if enroll_count==3
esttab out1 out2 out3 using t_B1_OLS_Final_Exam.rtf, replace  b(a3) t abs star(+ 0.10 * 0.05 ** 0.01 *** 0.001) r2(3) ar2(3) scalars (N "chi2 Wald Chi2" "p  Prob > chi2" F  "ll Log lik.") sfmt(3) nogaps label title ("Table B.1 OLS Regression Final Cumulative Exam by Delivery Modality")  
```

## 1. Reading and summarizing the data

In [3]:
dta_path = os.path.join(DATADIR, 'ReplicationData2.dta')
df = pd.read_stata(dta_path)

# using parentheses allows you to write your string on multiple lines
long_wrapped_str = textwrap.fill(str(list(df.columns)), 80)
print(f'Columns: {long_wrapped_str}')


# Note: the Stata code I'm replicating above all filters on enroll_count = 3. So I'll drop all of those observations
# First we'll print what that variable looks like
print_df = (                     # Parentheses let you break up commands in pythong
    df['enroll_count']           # filter to only look at variable enroll count
    .value_counts(dropna=False)  # value count, keeping na values
    .reset_index()               # reset index converst from series to dataframe (necessary for tabulate)
)
print('\nEnroll count values:')   # Note the \n at the start of the string; that prints to a new line
print(tabulate(print_df, headers='keys', showindex=False))

# drop values 
df = df.loc[df['enroll_count'] == 3]

Columns: ['gpa', 'cum_total_crds', 'gender', 'asian', 'black', 'hawaiian', 'hispanic',
'unknown', 'white', 'ethnic_dummy', 'format_ol', 'format_blended',
'sat_math_NEW', 'sat_verbal_NEW', 'enroll_count', 'format_f2f_v_ol',
'format_f2f_v_blended', 'format_combined_v_f2f', 'falsexam', 'experiment1']

Enroll count values:
  enroll_count    count
--------------  -------
             3      323
           nan      151
             2      102
             1       94


In [4]:
df.head()

,gpa,cum_total_crds,gender,asian,black,hawaiian,hispanic,unknown,white,ethnic_dummy,format_ol,format_blended,sat_math_NEW,sat_verbal_NEW,enroll_count,format_f2f_v_ol,format_f2f_v_blended,format_combined_v_f2f,falsexam,experiment1
151,NaN,53.0,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,540.0,500.0,3.0,0.0,0.0,1.0,63.299969,1.0
152,2.564,28.0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,640.0,470.0,3.0,0.0,0.0,1.0,79.959999,1.0
153,3.218,27.0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,560.0,530.0,3.0,NaN,1.0,0.0,83.370003,1.0
154,NaN,18.0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,670.0,660.0,3.0,NaN,1.0,0.0,90.019943,1.0
155,3.499,81.0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,660.0,630.0,3.0,1.0,NaN,0.0,83.300003,1.0


In [5]:
df.shape

(323, 20)

In [6]:
df[['format_ol', 'format_blended']].value_counts()

format_ol  format_blended
0.0        0.0               120
           1.0               109
1.0        0.0                94
Name: count, dtype: int64

## 2. Regression - Replicating Table 1 Panel A

### Panel A column 1

<img src="img/ACH16-table1_col1.png" width=300  />

Text describing the results from colum (1) of panel A:

>Panel A of Table 1 contains parameter estimates for differences in scores (out of a possible 100) on a cumulative final exam taken in a similar setting by all students. Column 1 contains estimates not including available covariates. As can be seen there, students in the purely online course score about 4.9 (f-statistic = 3.09) points worse than those in the face-to-face course.


Consider the first Stata regression:

```Stata
reg falsexam format_ol format_blended if enroll_count==3
```
As an equation:
\begin{equation}
\text{falsexam}_i = \beta_0 +  \beta_1 \cdot \text{format\_ol} + \beta_2 \cdot \text{format\_blended} + u_i
\end{equation}

In [7]:
ols_mod = smf.ols('falsexam ~ format_ol + format_blended', data=df)
ols_res = ols_mod.fit()

ols_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               falsexam   R-squared:                       0.030
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     4.922
Date:                Sun, 28 Sep 2025   Prob (F-statistic):            0.00785
Time:                        17:31:20   Log-Likelihood:                -1246.4
No. Observations:                 323   AIC:                             2499.
Df Residuals:                     320   BIC:                             2510.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         78.5475      1.052     74.654      0.000      76.477      80.617
format_ol         -4.9122      1.588     -3.094      0.002      -8.036      -1.789
format_blended    -1.4538      1.525     -0.953      0.341      -4.454       1.547
==============================================================================
Omnibus:                      158.483   Durbin-Watson:                   2.012
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1262.348
Skew:                          -1.861   Prob(JB):                    7.67e-275
Kurtosis:                      11.941   Cond. No.                         3.57
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

#### Creating exact replica of table

For this course, you should never worry about precisely matching the formatting of a particular table.
You do, however, need to be mindful that your results are readable.

* Acceptable: a summary table with some extraneous information (maybe <5 rows and/or <5 columns of extraneous information)
* Acceptable: a table where you have p-values and they have t-values
* Acceptable: a table where you round to inconsistent digits, and they round to three
* Unacceptable: a summary table with 50+ rows
* Unacceptable: a table with many rows, when you only need to read one coefficient
* Unacceptable: your results are in scientific notation that is difficult to interpret or read

That said, I'll include an example below about how you can precisely replicate results.

In [8]:
# Function to generate significance stars
def get_significance_stars(pval):
    if pval < 0.001:
        return '***'
    elif pval < 0.01:
        return '**'
    elif pval < 0.05:
        return '*'
    elif pval < 0.1:
        return '.'
    else:
        return ''

ols_res.pvalues.apply(get_significance_stars)

Intercept         ***
format_ol          **
format_blended       
dtype: object

In [9]:
keep_vars = ['format_ol', 'format_blended']

coef_series = (
    ols_res.params[keep_vars].round(3).astype(str)           # format the estimated parameters as rounded strings
    +                                                        # use addition to add stars (if any)
    ols_res.pvalues.apply(get_significance_stars)[keep_vars] # generate significance stars
)
coef_series.name = 'coef'  # naming series will be helpful when this is a dataframe
tvalue_series = '(' + ols_res.tvalues[keep_vars].round(2).astype(str) + ')' # add parentheses around formatted t-values
tvalue_series.name = 'tvalue'
obs_series = pd.Series([f'{ols_res.nobs:.0f}'], index=['_nobs'], name='obs')

ols_res_series = pd.concat([coef_series, tvalue_series, obs_series]).sort_index(ascending=False)
ols_res_series

format_ol         -4.912**
format_ol          (-3.09)
format_blended      -1.454
format_blended     (-0.95)
_nobs                  323
dtype: object

In [10]:
# combining the above into a function
def get_result_array(res, keep_vars, name=None):
    coef_series = (
        res.params[keep_vars].round(3).astype(str)           # format the estimated parameters as rounded strings
        +                                                        # use addition to add stars (if any)
        res.pvalues.apply(get_significance_stars)[keep_vars] # generate significance stars
    )
    coef_series.name = 'coef'  # naming series will be helpful when this is a dataframe
    tvalue_series = '(' + res.tvalues[keep_vars].round(2).astype(str) + ')' # add parentheses around formatted t-values
    tvalue_series.name = 'tvalue'
    obs_series = pd.Series([f'{res.nobs:.0f}'], index=['_nobs'], name='obs')

    ols_res_series = pd.concat([coef_series, tvalue_series, obs_series]).sort_index(ascending=False)
    if name is not None:
        ols_res_series.name = name
    
    return ols_res_series

ols_col1 = get_result_array(ols_res, keep_vars=['format_ol', 'format_blended'], name='(1)')
ols_col1

format_ol         -4.912**
format_ol          (-3.09)
format_blended      -1.454
format_blended     (-0.95)
_nobs                  323
Name: (1), dtype: object

### Panel A column 2

<img src="img/ACH16-table1_col2.png" width=300  />

Text describing the results from column (2) of panel A:

>In column 2, covariates are added, most related to learning outcomes (prior GPA, prior credits, and SAT scores). With these controls, students in the purely online course are still found to score 5.2 ($t$-statistic = 3.26) points lower on the final exam. 

S
tata command:

```Stata
eststo out2: reg falsexam format_ol format_blended gpa cum_total_crds sat_math_NEW sat_verbal_NEW if enroll_count==3
```

In [11]:
ols_mod2 = smf.ols('falsexam ~ format_ol + format_blended + gpa + cum_total_crds + sat_math_NEW + sat_verbal_NEW', data=df)
ols_res2 = ols_mod2.fit()

ols_res2.summary().tables[1]
ols_res2.summary(slim=True)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               falsexam   R-squared:                       0.234
Model:                            OLS   Adj. R-squared:                  0.216
No. Observations:                 265   F-statistic:                     13.11
Covariance Type:            nonrobust   Prob (F-statistic):           6.09e-13
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         41.1684      7.429      5.541      0.000      26.538      55.799
format_ol         -5.2005      1.596     -3.258      0.001      -8.344      -2.057
format_blended    -1.7032      1.524     -1.118      0.265      -4.703       1.297
gpa                7.7024      1.221      6.307      0.000       5.297      10.107
cum_total_crds    -0.0453      0.033     -1.378      0.169      -0.110       0.019
sat_math_NEW       0.0346      0.010      3.637      0.000       0.016       0.053
sat_verbal_NEW    -0.0111      0.009     -1.215      0.225      -0.029       0.007
==================================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.02e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [12]:
ols_col2 = get_result_array(ols_res2, keep_vars=['format_ol', 'format_blended'], name='(2)')

### Panel A column 3

<img src="img/ACH16-table1_col3.png" width=300  />

Text describing the results from column (3) of panel A:

>Column 3 includes all available covariates. There, students are still found to score 4.2 (t-statistic = 2.68) points lower in the online section than the face-to-face variant. The sign of the impact of participating in the blended course is negative but the parameters are not significantly different than zero at conventional levels across the three columns.

Stata code:
```Stata
reg falsexam format_ol format_blended gpa cum_total_crds sat_math_NEW sat_verbal_NEW gender asian black hispanic hawaiian unknown if enroll_count==3
```

In [13]:
reg_str = (
    'falsexam ~ format_ol + format_blended + gpa + cum_total_crds + sat_math_NEW + sat_verbal_NEW'
    ' + gender + asian + black + hispanic + hawaiian + unknown'
)
print(reg_str)
ols_mod3 = smf.ols(reg_str, data=df)
ols_res3 = ols_mod3.fit()

ols_res3.summary(slim=True)

falsexam ~ format_ol + format_blended + gpa + cum_total_crds + sat_math_NEW + sat_verbal_NEW + gender + asian + black + hispanic + hawaiian + unknown


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               falsexam   R-squared:                       0.345
Model:                            OLS   Adj. R-squared:                  0.307
No. Observations:                 215   F-statistic:                     8.883
Covariance Type:            nonrobust   Prob (F-statistic):           1.42e-13
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         39.9245      8.433      4.734      0.000      23.297      56.553
format_ol         -4.2324      1.578     -2.682      0.008      -7.345      -1.120
format_blended    -0.9959      1.511     -0.659      0.511      -3.976       1.984
gpa                8.5803      1.238      6.933      0.000       6.140      11.021
cum_total_crds    -0.0483      0.033     -1.480      0.140      -0.113       0.016
sat_math_NEW       0.0246      0.010      2.346      0.020       0.004       0.045
sat_verbal_NEW    -0.0073      0.010     -0.770      0.442      -0.026       0.011
gender             4.6438      1.358      3.421      0.001       1.967       7.321
asian             -0.9728      1.717     -0.567      0.572      -4.359       2.413
black             -3.3353      2.575     -1.295      0.197      -8.413       1.742
hispanic           1.0430      4.734      0.220      0.826      -8.292      10.378
hawaiian           6.0556      5.509      1.099      0.273      -4.807      16.919
unknown          -10.9507      9.555     -1.146      0.253     -29.792       7.891
==================================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.37e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [14]:
ols_col3 = get_result_array(ols_res3, keep_vars=['format_ol', 'format_blended'], name='(3)')

In [15]:
result_df = pd.concat([ols_col1, ols_col2, ols_col3], axis=1)

# rename index values
result_df.rename(index={
    'format_ol': 'Dummy online = 1',
    'format_blended': 'Dummy blended = 1',
    '_nobs': 'Observations'}, inplace=True)

print(tabulate(result_df, headers='keys'))

                   (1)       (2)       (3)
-----------------  --------  --------  --------
Dummy online = 1   -4.912**  -5.201**  -4.232**
Dummy online = 1   (-3.09)   (-3.26)   (-2.68)
Dummy blended = 1  -1.454    -1.703    -0.996
Dummy blended = 1  (-0.95)   (-1.12)   (-0.66)
Observations       323       265       215


## 3. Regression and matching note

In [16]:
result = smf.ols('falsexam ~ format_ol', data=df).fit()
result.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,77.8555,0.762,102.235,0.000,76.357,79.354
format_ol,-4.2203,1.412,-2.990,0.003,-6.998,-1.443


In [17]:
df.groupby(['format_ol'])['falsexam'].mean()

format_ol
0.0    77.855522
1.0    73.635262
Name: falsexam, dtype: float32

In [18]:
df.groupby(['format_ol'])['falsexam'].mean().diff()

format_ol
0.0         NaN
1.0   -4.220261
Name: falsexam, dtype: float32

In [19]:
result = smf.ols('falsexam ~ format_ol + format_blended', data=df).fit()
result.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,78.5475,1.052,74.654,0.000,76.477,80.617
format_ol,-4.9122,1.588,-3.094,0.002,-8.036,-1.789
format_blended,-1.4538,1.525,-0.953,0.341,-4.454,1.547


In [20]:
mean_df = df.groupby(['format_ol', 'format_blended'])['falsexam'].mean()
mean_df

format_ol  format_blended
0.0        0.0               78.547485
           1.0               77.093735
1.0        0.0               73.635262
Name: falsexam, dtype: float32

In [21]:
mean_df.loc[:, 0]
mean_df.loc[:, 0].diff()

format_ol
0.0         NaN
1.0   -4.912224
Name: falsexam, dtype: float32

In [22]:
mean_df.loc[0, :]
mean_df.loc[0, :].diff()

format_blended
0.0         NaN
1.0   -1.453751
Name: falsexam, dtype: float32